# 08 — Auditoría y reconciliación de resultados

**Caso Tostao | Supply Chain Analytics**

Este notebook cierra la diferencia histórica del pedido Q90. Ejecuta el pipeline modular, reproduce el cálculo de Notebook 04 con los mismos insumos y compara ambos resultados.

> Ejecutar desde la raíz del repositorio, con los cinco CSV en `data/`. No interpreta el pedido sugerido como una orden de compra real.

In [1]:
from pathlib import Path
import sys, hashlib
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "tostao_supply_chain").exists():
    raise FileNotFoundError("Ejecuta este notebook desde la raíz del proyecto.")
sys.path.insert(0, str(ROOT / "src"))

from tostao_supply_chain.pipeline import run_pipeline
from tostao_supply_chain.config import PipelineConfig

DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

required = [
    "ventas_historicas.csv", "inventario_actual.csv",
    "catalogo_productos.csv", "maestro_tiendas.csv",
    "backtesting_modelos_base.csv"
]
missing = [f for f in required if not (DATA / f).exists()]
if missing:
    raise FileNotFoundError(f"Faltan archivos en data/: {missing}")

print("Entradas completas:", len(required), "de", len(required))

Entradas completas: 5 de 5


In [2]:
# 1. Ejecutar pipeline modular (Notebook 07)
config = PipelineConfig(random_seed=42)
run_summary, _ = run_pipeline(DATA, OUT, config=config)

forecast_daily = pd.read_csv(OUT / "forecast_ets_diario_7d.csv")
forecast_weekly = pd.read_csv(OUT / "forecast_ets_semanal_sku_tienda.csv")
weekly_errors = pd.read_csv(OUT / "errores_semanales_backtesting_ets.csv")
recommendations = pd.read_csv(OUT / "recomendacion_abastecimiento_Q90.csv")
comparison = pd.read_csv(OUT / "comparacion_politicas_Q85_Q90_Q95.csv")

run_summary

{'fecha_ultimo_dato': '2024-03-31',
 'horizonte_inicio': '2024-04-01',
 'horizonte_fin': '2024-04-07',
 'combinaciones_sku_tienda': 160,
 'observaciones_ventas': 14560,
 'residuos_backtesting': 480,
 'combinaciones_con_pedido': 156,
 'unidades_pedido_Q90': 13981,
 'fallbacks_forecast': 0}

## 2. Reproducción independiente de Notebook 04

Se vuelve a ejecutar el remuestreo de errores relativos ETS por producto, con semilla 42, 10.000 simulaciones por SKU–tienda, Q90 y tope de cobertura de ocho días. Esta reproducción permite comprobar que el resultado no depende exclusivamente de la función `run_pipeline`.

In [3]:
sales = pd.read_csv(DATA / "ventas_historicas.csv")
inventory = pd.read_csv(DATA / "inventario_actual.csv")
products = pd.read_csv(DATA / "catalogo_productos.csv")
backtest = pd.read_csv(DATA / "backtesting_modelos_base.csv")

keys = ["id_tienda", "id_producto"]
bt_ets = backtest[backtest["modelo"] == "ETS_Aditivo_7"].copy()
bt_ets["fecha"] = pd.to_datetime(bt_ets["fecha"])

errors = (bt_ets.groupby(keys + ["fold"], as_index=False)
          .agg(demanda_real=("y", "sum"),
               demanda_predicha=("pred", "sum"),
               dias_evaluados=("fecha", "nunique")))
errors["error_relativo"] = (
    (errors["demanda_real"] - errors["demanda_predicha"])
    / errors["demanda_predicha"].clip(lower=1)
)
errors = errors.replace([np.inf, -np.inf], np.nan).dropna(subset=["error_relativo"])

rng = np.random.default_rng(42)
global_pool = errors["error_relativo"].to_numpy()
product_pools = {
    product: group["error_relativo"].to_numpy()
    for product, group in errors.groupby("id_producto")
}

direct_orders = []
for _, row in forecast_weekly.iterrows():
    pool = product_pools.get(row["id_producto"], np.array([]))
    if len(pool) < 30:
        pool = global_pool
    sampled = rng.choice(pool, size=10_000, replace=True)
    scenarios = np.maximum(
        0, float(row["demanda_pronosticada_7d"]) * (1 + sampled)
    )
    q90 = float(np.quantile(scenarios, 0.90))
    daily_mean = max(0.0, float(row["demanda_pronosticada_7d"])) / 7
    cap_8d = int(np.ceil(daily_mean * 8))
    target = min(int(np.ceil(q90)), cap_8d)

    stock = inventory.loc[
        (inventory["id_tienda"] == row["id_tienda"]) &
        (inventory["id_producto"] == row["id_producto"]),
        "stock_actual"
    ].iloc[0]
    direct_orders.append(max(0, int(np.ceil(target - stock))))

direct_q90_total = int(sum(direct_orders))
pipeline_q90_total = int(recommendations["pedido_sugerido"].sum())

print("Notebook 04 reproducido:", f"{direct_q90_total:,}", "unidades")
print("Pipeline 07:", f"{pipeline_q90_total:,}", "unidades")
assert direct_q90_total == pipeline_q90_total, "Los dos métodos no coinciden." 

Notebook 04 reproducido: 13,981 unidades
Pipeline 07: 13,981 unidades


## 3. Controles de integridad

Se comprueban las llaves, el horizonte, la suma del pronóstico diario, la correspondencia con inventario y el cálculo del pedido.

In [4]:
daily_sum = (forecast_daily.groupby(keys, as_index=False)
             .agg(demanda_diaria_total=("demanda_pronosticada", "sum")))
check_forecast = daily_sum.merge(forecast_weekly, on=keys, validate="one_to_one")
max_forecast_diff = (
    check_forecast["demanda_diaria_total"]
    - check_forecast["demanda_pronosticada_7d"]
).abs().max()

stock_match = recommendations.merge(
    inventory[keys + ["stock_actual"]], on=keys,
    suffixes=("_rec", "_src"), validate="one_to_one"
)
controls = [
    ("5 archivos de entrada presentes", len(required) == 5),
    ("160 combinaciones SKU-tienda", len(forecast_weekly) == 160),
    ("480 errores semanales ETS", len(weekly_errors) == 480),
    ("Q90 contiene 160 combinaciones", len(recommendations) == 160),
    ("156 combinaciones con pedido", int((recommendations.pedido_sugerido > 0).sum()) == 156),
    ("Notebook 04 coincide con pipeline", direct_q90_total == pipeline_q90_total),
    ("Pronóstico diario suma el semanal", max_forecast_diff < 1e-8),
    ("Stock recomendado coincide con fuente", np.allclose(stock_match.stock_actual_rec, stock_match.stock_actual_src)),
    ("Pedido no negativo", (recommendations.pedido_sugerido >= 0).all()),
    ("Pedido = objetivo menos stock", (recommendations.pedido_sugerido == np.maximum(0, np.ceil(recommendations.inventario_objetivo - recommendations.stock_actual))).all()),
    ("3 políticas por combinación", len(comparison) == 480 and comparison.groupby(keys).size().eq(3).all()),
    ("Sin fallback ETS", (forecast_daily.modelo == "ETS_Aditivo_7").all()),
]
audit = pd.DataFrame(controls, columns=["control", "aprobado"])
audit["estado"] = np.where(audit["aprobado"], "OK", "REVISAR")
display(audit)
assert audit["aprobado"].all(), "Hay controles que requieren revisión."

audit.to_csv(OUT / "auditoria_controles_reconciliada.csv", index=False)

,control,aprobado,estado
0,5 archivos de entrada presentes,True,OK
1,160 combinaciones SKU-tienda,True,OK
2,480 errores semanales ETS,True,OK
3,Q90 contiene 160 combinaciones,True,OK
4,156 combinaciones con pedido,True,OK
5,Notebook 04 coincide con pipeline,True,OK
6,Pronóstico diario suma el semanal,True,OK
7,Stock recomendado coincide con fuente,True,OK
8,Pedido no negativo,True,OK
9,Pedido = objetivo menos stock,True,OK


## 4. Trazabilidad de insumos

Los hashes SHA-256 permiten identificar exactamente los archivos usados en esta ejecución.

In [5]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hashes = pd.DataFrame([
    {"archivo": name, "sha256": sha256(DATA / name)}
    for name in required
])
display(hashes)
hashes.to_csv(OUT / "hashes_insumos_08.csv", index=False)

print("\nRESULTADO CANÓNICO Q90:", f"{pipeline_q90_total:,}", "unidades")
print("Combinaciones SKU-tienda:", len(recommendations))
print("Combinaciones con pedido:", int((recommendations.pedido_sugerido > 0).sum()))
print("Corte de ventas:", run_summary["fecha_ultimo_dato"])
print("Horizonte:", run_summary["horizonte_inicio"], "a", run_summary["horizonte_fin"])

,archivo,sha256
0,ventas_historicas.csv,375749b680875d27987150abdcc4deb91dc32fd584f16e...
1,inventario_actual.csv,181d33a98f5a0e614c162120f33151eeaa6b355b3c89ea...
2,catalogo_productos.csv,ee9ef3dc586e99c8890c063c44c08281ed3ea201201722...
3,maestro_tiendas.csv,127e3a3b05d72e2ce7dda4b3e7af861ce4b59d4908475d...
4,backtesting_modelos_base.csv,ce5fb8c635f94f781f873bef2824ca8baa28121ff71626...



RESULTADO CANÓNICO Q90: 13,981 unidades
Combinaciones SKU-tienda: 160
Combinaciones con pedido: 156
Corte de ventas: 2024-03-31
Horizonte: 2024-04-01 a 2024-04-07


## 5. Interpretación

Con los archivos actuales, la reproducción independiente de Notebook 04 y el pipeline 07 coinciden en **13.981 unidades Q90**. Los totales históricos de 13.805 y 13.972 no se reproducen con esta versión de insumos/código; no se atribuye una causa específica sin disponer de los artefactos exactos de esas ejecuciones.

La cifra es una recomendación de pedido bajo supuestos simplificados, no una orden real. El modelo no dispone de lead time, pedidos en tránsito, recepciones, múltiplos de compra, vida útil ni capacidad física de almacenamiento. El tope de ocho días es un supuesto de política.